# COCO Human Detection Training

Use this notebook to download COCO data, train a MobileNetV2 model, and convert it for ESP32.

In [ ]:
# Install Dependencies
!pip install -q fiftyone tensorflow
import fiftyone as fo
import fiftyone.zoo as foz
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# Download COCO Data (Person vs Background)

# 1. Download 2000 "Person" images
print("Downloading 'Person' images...")
dataset_person = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    label_types=["detections"],
    classes=["person"],
    max_samples=2000,
    shuffle=True,
    dataset_name="coco-person"
)

# 2. Download 2000 "Background" images
print("Downloading 'Background' images...")
dataset_background = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    label_types=["detections"],
    classes=["cat", "dog", "car", "chair", "potted plant"],
    max_samples=2000,
    shuffle=True,
    dataset_name="coco-background"
)

In [ ]:
# Prepare Dataset
BASE_DIR = "coco_dataset_formatted"
if os.path.exists(BASE_DIR): shutil.rmtree(BASE_DIR)
os.makedirs(f"{BASE_DIR}/person")
os.makedirs(f"{BASE_DIR}/background")

print("Organizing images...")

for sample in dataset_person:
    shutil.copy(sample.filepath, f"{BASE_DIR}/person/{os.path.basename(sample.filepath)}")

for sample in dataset_background:
    shutil.copy(sample.filepath, f"{BASE_DIR}/background/{os.path.basename(sample.filepath)}")

print(f"Dataset Ready: {len(os.listdir(f'{BASE_DIR}/person'))} Persons, {len(os.listdir(f'{BASE_DIR}/background'))} Backgrounds")

In [ ]:
# Load & Train MobileNetV2
IMG_SIZE = 96
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
  BASE_DIR, validation_split=0.2, subset="training", seed=123,
  image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
  BASE_DIR, validation_split=0.2, subset="validation", seed=123,
  image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
)

# Model: MobileNetV2 (Alpha 0.35)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, alpha=0.35, weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
  layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
  base_model,
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.2),
  layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
history = model.fit(train_ds, validation_data=val_ds, epochs=15)

In [ ]:
# Convert to ESP32 (Quantized)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
  for input_value, _ in train_ds.take(100):
    yield [tf.cast(input_value, tf.float32)]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

# Save as Header File
with open('model_data.h', 'w') as f:
    f.write('const unsigned char g_model[] = {')
    f.write(','.join([str(i) for i in tflite_model]))
    f.write('};\n')
    f.write(f'const int g_model_len = {len(tflite_model)};')

print("Done! Check model_data.h")